Code to set path root

In [ ]:
import sys
import os
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


sys.path.append(os.path.abspath(".."))

# Training model on `fight-weaponized-other-dataset` with 64x64 Image Sizes
* using `datasets`, `transforms` module from `torchvison`
* using `dataloader` module from `torch.utils.data`

Extention epoch of epxperiment 4 as it shows possible grow space and tends

## Importing necessary Modules

In [ ]:
# Import torch libraries
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn

# Import modules
from modules.architectures.Architecture import Architecture, ResidualBlock
from modules.helper.Trainer import Trainer
from modules.helper.Plotter import plot_training_metrics, plot_testing_history
from modules.helper.Tester import  Tester

Check if CUDA is used

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))
    print("Current device index:", torch.cuda.current_device())
    print("Device count:", torch.cuda.device_count())
else:
    print("Running on CPU")

### Use datasets, dataloader and transforms for loading training Dataset

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])
train_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/train",
    transform = train_transform
)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    shuffle=True
)

print("Total Batches => ", len(train_dataloader))

### Use datasets, dataloader and transforms for loading validation Dataset

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

val_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/val",
    transform = val_transform
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=32,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Total Batches => ", len(val_dataloader))

### Using Model Architecture:
* 10 Convolutional Layers
    - Conv2D
    - BatchNorm2D
    - ReLu
    - MaxPool2D (Optional)
* 1 Linear Layer
* SDG Optimizer

In [ ]:
model = Architecture().to("cuda")

### Adding 8 blocks (MaxPool2D in each second block)

In [ ]:
in_channels = 3
out_channels = 8
size = 64

model_blocks = []

for i in range(1, 9):
    conv = nn.Conv2d(in_channels, out_channels, 3, 1, 1)
    batch_norm = nn.BatchNorm2d(out_channels)
    model_blocks.extend(
        [conv, batch_norm, nn.ReLU()]
    )
    if i%2==0:
        model_blocks.append(nn.MaxPool2d(2,2))
        size = size//2
    
    in_channels = out_channels
    out_channels = out_channels * 2

print(f"Final In Channels = {in_channels}")
print(f"Final Out Channels = {out_channels}")
print(f"Final Shape = {size}")

In [ ]:
model = model.add(
    # Conv Blocks
    *model_blocks,
    
    # Flatten
    nn.Flatten(),

    nn.Linear(in_channels * size * size, in_channels),
    nn.ReLU(),
    nn.Linear(in_channels, 128),
    nn.ReLU(),
    nn.Linear(128, 3)
    )

In [ ]:
checkpoint = torch.load("../models/experiment4/model_epoch_97.pt")

model.load_state_dict(checkpoint["model"])

### Use Trainer to train and check validations
Adding weight decay and decreased weight

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=3e-4, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
trainer = Trainer(
    model, 
    train_dataloader, 
    val_dataloader, 
    optimizer=optimizer, 
    num_classes=3,
    criterion=criterion,
    device="cuda",
    save_dir="../models/experiment10/",
    save_checkpoints=1,
    print_every=10
    )

In [ ]:
history = trainer.fit(100)

### Save Metrics

In [ ]:
df = plot_training_metrics(history)
df.to_csv("../documentations/experiments/experiment10/tables/training_metrics.csv", index=False)

In [ ]:
filtered_df = df[
    (df["val_accuracy"] >= 0.8) &
    (df["val_f1"] >= 0.8)
].to_dict()

high_accuracy_df = plot_training_metrics(filtered_df)

### Training/Validation Trend (100 epochs)
* Train loss steadily decreases from ~0.13 → ~0.017, indicating very strong optimization convergence.
* Train accuracy increases rapidly and saturates extremely early (>0.98 by epoch ~2–5), later reaching ~1.0 for long stretches.
* Train precision/recall/F1 also saturate near-perfect values (>0.99) from early epochs onward.
* Validation accuracy improves from ~0.81 → peak ~0.85 range, then oscillates.
* Validation loss is noisy and does not consistently decrease after early epochs; it fluctuates significantly.
* Strong generalization gap appears after early training (train ≈ 1.0 vs val ≈ 0.83–0.85).
* Best validation performance occurs in a mid-to-late window, not at final epochs.
* After peak performance, validation metrics degrade or oscillate while training remains perfect → overfitting.
* Several spikes in validation loss (e.g., epochs ~31, 46, 79, 94, 97) indicate instability or data sensitivity.
* Confusion matrices remain relatively stable in structure, but misclassification patterns persist across epochs.
* Later epochs show diminishing returns and occasional catastrophic validation drops despite perfect train metrics.

The model converges very quickly on the training set, reaching near-perfect performance (train accuracy/F1 ≈ 0.99–1.0) and continuously reducing train loss from ~0.13 to ~0.017, while validation performance improves only up to a plateau and then oscillates; validation accuracy rises from ~0.81 to a stable range around ~0.83–0.85 with the best result around epoch 78 (val_accuracy ≈ 0.8496, val_f1 ≈ 0.8511), after which further training does not improve generalization and instead introduces instability and occasional sharp validation loss spikes, indicating clear overfitting where additional epochs mainly improve memorization rather than test performance.

<b>Best Epoch 78</b>

<b>Loss</b>
* Train Loss = 0.02864
* Valid Loss = 0.61732
* 
<b>Training Metrics</b>
* Train Accuracy = 0.99953
* Train Precison = 0.99950
* Train Recall = 0.99955
* Train F1 = 0.99952

<b>Validation Accuracy</b>
* Validation Accuracy = 0.84956
* Validation Precision = 0.85459
* Validation Recall = 0.84930
* Validation F1 = 0.85105

## Use Tester Module to Test Model

Load Model with State Dict

In [ ]:
# Transforms of Data
test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# Dataset Loading From Image dir
test_dataset = datasets.ImageFolder(
    root="../datasets/fight-weaponized-other-dataset/test", 
    transform = test_transform 
    )

# DataLoader
test_loader = DataLoader(
    dataset=test_dataset, 
    batch_size=64,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
    )

tester = Tester(
    model,
    test_loader,
    3,
    torch.nn.CrossEntropyLoss(),
    "cuda"
)

test_scores = tester.test_all_checkpoints(
    "../models/experiment10"
)

### Save Test Metrics

In [ ]:
# Plot all 100 epochs
test_metrics_df = plot_testing_history(test_scores)
test_metrics_df.to_csv("../documentations/experiments/experiment10/tables/test_metrics.csv", index=False)

### Test Performance Trend (100 epochs)
* Test accuracy improves quickly from ~0.82 to ~0.84 in early epochs, then stabilizes in a narrow band (~0.84–0.87) for most of training
* Test loss decreases early (≈0.50 → ≈0.43–0.47 range) but remains noisy throughout, with repeated spikes (notably around epochs 8, 12, 31, 46, 79–80, 97).
* Model shows consistent generalization in the mid-training region (roughly epochs 20–75), where metrics are stable and tightly clustered.
* Best performance region occurs around epochs 40–60, where accuracy and F1 repeatedly peak (~0.86–0.87).
* Performance degrades sharply in isolated epochs despite good neighboring epochs, indicating instability rather than gradual drift.
* Late training does not improve performance; instead, it introduces larger variance in loss and occasional drops in accuracy/F1.
* Precision, recall, and F1 track closely throughout, indicating balanced classification behavior with no strong bias shift.
* No clear overfitting trend in test accuracy itself, but increasing volatility suggests sensitivity to optimization dynamics in later epochs.
* Model is effectively converged by mid-training; later epochs mostly oscillate around the same performance ceiling without consistent gains.

Test performance shows stable convergence with most epochs clustering in a consistent range (accuracy ~0.83–0.87, F1 ~0.83–0.87), indicating good generalization overall with occasional instability. The model reaches its best region around the mid-to-late training phase, where both loss and classification metrics are well-balanced, but several spikes in test loss (e.g., epochs 31, 79–80, 97) indicate periodic instability and overfitting bursts despite generally strong performance. The most reliable high-performance window is around epochs 43–86, where accuracy and F1 consistently peak and remain stable without extreme loss spikes.

<b>Best Epoch 44</b>

* Loss = 0.43377
* Accuracy = 0.87061
* Precision = 0.87164
* Recall = 0.87177
* F1-Score = 0.87158